# Notebook 2 - Datasheet Retrieval (Qdrant hybrid search)

Searches the collection built by `01_ingestion.ipynb`.

1. **Configuration** - question, collection, top-k
2. **Connect** to Qdrant and list the ingested datasheets
3. **Load** the same embedding model used for ingestion
4. **Search** - dense, sparse (keyword) and **hybrid** (Reciprocal Rank Fusion of both), with optional part-number filter
5. **Compare** the three modes side by side
6. **Build a cited context** for an LLM, and optionally generate an answer

## 1. Configuration

In [13]:
from pathlib import Path

QUESTION = "What is the quiescent current in shutdown?"
TOP_K = 5

# ---- must match 01_ingestion.ipynb ----
QDRANT_MODE = "server"            # "server" or "local"
QDRANT_URL = "http://localhost:6333"
QDRANT_API_KEY = None
COLLECTION = "semiconductor_datasheets"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"

AUTO_PART_FILTER = True           # if the question names an ingested part (e.g. LTC3310S), only search chunks for it
PREFETCH_MULTIPLIER = 4           # hybrid: candidates taken from each of dense / sparse = TOP_K * this

PROJECT_DIR = Path.cwd()

## 2. Imports and connect to Qdrant

In [14]:
import os
import re
import time
import zlib
from collections import Counter

import pandas as pd
from IPython.display import Markdown, display
from qdrant_client import QdrantClient, models

try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
pd.set_option("display.max_colwidth", 140)


def connect_qdrant(timeout_s=120, poll_s=5):
    if QDRANT_MODE == "local":
        return QdrantClient(path=str(PROJECT_DIR / "data" / "qdrant_local"))
    deadline = time.time() + timeout_s
    while True:
        try:
            client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=10)
            client.get_collections()
            return client
        except Exception as exc:
            if time.time() > deadline:
                raise RuntimeError(f"Qdrant not reachable at {QDRANT_URL}") from exc
            print(f"Waiting for Qdrant at {QDRANT_URL} ...")
            time.sleep(poll_s)


client = connect_qdrant()
assert client.collection_exists(COLLECTION), f"Collection '{COLLECTION}' not found - run 01_ingestion.ipynb first"
print(f"Connected. '{COLLECTION}' holds {client.count(COLLECTION, exact=True).count} chunks")

Connected. 'semiconductor_datasheets' holds 134 chunks


In [15]:
# Which datasheets are in the collection?
docs, offset = {}, None
while True:
    batch, offset = client.scroll(COLLECTION, limit=512, offset=offset, with_vectors=False,
                                  with_payload=["doc_sha", "file_name", "title", "doc_parts", "manufacturer",
                                                "revision", "embedding_model", "ingested_at"])
    for p in batch:
        d = docs.setdefault(p.payload["doc_sha"], {**p.payload, "chunks": 0})
        d["chunks"] += 1
    if offset is None:
        break

docs_df = pd.DataFrame(docs.values())[["file_name", "title", "doc_parts", "manufacturer", "revision", "chunks",
                                       "embedding_model", "ingested_at"]]
KNOWN_PARTS = sorted({p for d in docs.values() for p in d["doc_parts"]}, key=len, reverse=True)
models_used = set(docs_df.embedding_model)
assert models_used == {EMBED_MODEL}, f"Collection was embedded with {models_used}, notebook uses {EMBED_MODEL}"
docs_df

,file_name,title,doc_parts,manufacturer,revision,chunks,embedding_model,ingested_at
0,ltc3310s-3310s-1-3310-3310-1.pdf,LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 (Rev. F),"[LTC3310S, LTC3310S-1, LTC3310, LTC3310-1]",Analog Devices,F,134,BAAI/bge-small-en-v1.5,2026-11-05T12:58:12+00:00


## 3. Load the embedding model and the keyword vectoriser

In [16]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
# bge v1.5 English models were trained with this instruction on queries (not on documents)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: " if "bge-" in EMBED_MODEL and "-en-" in EMBED_MODEL else ""


def dense_query(text):
    return embedder.encode(QUERY_PREFIX + text, normalize_embeddings=True).tolist()


# ---- keep identical to 01_ingestion.ipynb (step 5c) ----
TOKEN_RE = re.compile(r"[a-z0-9µμ]+(?:[.\-/_][a-z0-9µμ]+)*")
STOPWORDS = set("a an and are as at be by for from has have in is it its of on or that the this to was were will with".split())


def sparse_terms(text):
    terms = []
    for tok in TOKEN_RE.findall(text.lower()):
        terms.append(tok)
        if re.search(r"[.\-/_]", tok):
            terms += [t for t in re.split(r"[.\-/_]", tok) if t]
    return [t for t in terms if t not in STOPWORDS]


def term_index(term):
    return zlib.crc32(term.encode("utf-8")) & 0x7FFFFFFF
# ---------------------------------------------------------


def sparse_query(text):
    """Query side: each distinct term weight 1; Qdrant applies IDF from the collection."""
    idx = sorted({term_index(t) for t in sparse_terms(text)})
    return models.SparseVector(indices=idx, values=[1.0] * len(idx))


print(f"Model ready. Known parts: {KNOWN_PARTS}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model ready. Known parts: ['LTC3310S-1', 'LTC3310-1', 'LTC3310S', 'LTC3310']


## 4. Search functions

- `dense`  - semantic similarity on the `dense` vector
- `sparse` - keyword match on the `sparse` vector (IDF-weighted)
- `hybrid` - both run as prefetches, merged with **Reciprocal Rank Fusion** inside Qdrant

Filters: part numbers (auto-detected from the question), a specific file, or chunk type (`text` / `table` / `caption`).

In [17]:
def parts_in_question(question):
    """Ingested part numbers mentioned in the question; 'LTC3310SEV#PBF' still matches LTC3310S."""
    q = question.upper()
    found = []
    for part in KNOWN_PARTS:                       # longest first, so LTC3310S-1 wins over LTC3310S
        if re.search(rf"(?<![A-Z0-9]){re.escape(part)}", q) and not any(part in f for f in found):
            found.append(part)
    return found


def build_filter(parts=None, file_name=None, chunk_type=None):
    must = []
    if parts:
        must.append(models.FieldCondition(key="parts", match=models.MatchAny(any=list(parts))))
    if file_name:
        must.append(models.FieldCondition(key="file_name", match=models.MatchValue(value=file_name)))
    if chunk_type:
        must.append(models.FieldCondition(key="type", match=models.MatchValue(value=chunk_type)))
    return models.Filter(must=must) if must else None


def search(question, mode="hybrid", top_k=TOP_K, parts=None, file_name=None, chunk_type=None):
    if parts is None and AUTO_PART_FILTER:
        parts = parts_in_question(question)
    flt = build_filter(parts, file_name, chunk_type)
    if mode == "dense":
        res = client.query_points(COLLECTION, query=dense_query(question), using="dense",
                                  query_filter=flt, limit=top_k, with_payload=True)
    elif mode == "sparse":
        res = client.query_points(COLLECTION, query=sparse_query(question), using="sparse",
                                  query_filter=flt, limit=top_k, with_payload=True)
    elif mode == "hybrid":
        n = top_k * PREFETCH_MULTIPLIER
        res = client.query_points(
            COLLECTION,
            prefetch=[
                models.Prefetch(query=dense_query(question), using="dense", filter=flt, limit=n),
                models.Prefetch(query=sparse_query(question), using="sparse", filter=flt, limit=n),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=top_k, with_payload=True,
        )
    else:
        raise ValueError("mode must be 'dense', 'sparse' or 'hybrid'")
    return res.points


def oneline(text, n):
    return re.sub(r"\s+", " ", text)[:n]


def results_table(points):
    return pd.DataFrame([{
        "rank": i + 1,
        "score": round(p.score, 4),
        "file": p.payload["file_name"],
        "page": p.payload["page"],
        "type": p.payload["type"],
        "section": p.payload["section_path"][-60:],
        "preview": oneline(p.payload["text"], 140),
    } for i, p in enumerate(points)])


def show(points, full_text=True):
    display(results_table(points))
    if full_text:
        for i, p in enumerate(points, 1):
            display(Markdown(f"**[{i}] {p.payload['file_name']} - page {p.payload['page']} - {p.payload['type']}** "
                             f"(score {p.score:.4f})\n\n```\n{p.payload['text']}\n```"))

## 5. Ask a question

In [18]:
print("Question:", QUESTION)
print("Part filter:", parts_in_question(QUESTION) if AUTO_PART_FILTER else "off")
hits = search(QUESTION, mode="hybrid")
show(hits)

Question: What is the quiescent current in shutdown?
Part filter: []


,rank,score,file,page,type,section,preview
0,1,1.0000,ltc3310s-3310s-1-3310-3310-1.pdf,3,table,ELECTRICAL CHARACTERISTICS,"[ELECTRICAL CHARACTERISTICS] VIN Quiescent Current in Shutdown - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 3 (table p3-t1, p..."
1,2,0.6667,ltc3310s-3310s-1-3310-3310-1.pdf,3,table,ELECTRICAL CHARACTERISTICS,"[ELECTRICAL CHARACTERISTICS] VIN Quiescent Current - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 3 (table p3-t1, page 3) | Par..."
2,3,0.2976,ltc3310s-3310s-1-3310-3310-1.pdf,4,table,ELECTRICAL CHARACTERISTICS,"[ELECTRICAL CHARACTERISTICS] Top Switch Current Limit - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 4 (table p4-t1, page 4) | ..."
3,4,0.2576,ltc3310s-3310s-1-3310-3310-1.pdf,19,text,APPLICATIONS INFORMATION > Enable Threshold Programming,[APPLICATIONS INFORMATION > Enable Threshold Programming] The LTC3310S has a precision threshold enable pin to enable or disable switchi...
4,5,0.2500,ltc3310s-3310s-1-3310-3310-1.pdf,12,text,OPERATION > Voltage Regulation,[OPERATION > Voltage Regulation] The internal error amplifier senses the difference between this feedback voltage and a 0.5V AGND refere...


**[1] ltc3310s-3310s-1-3310-3310-1.pdf - page 3 - table** (score 1.0000)

```
[ELECTRICAL CHARACTERISTICS]
VIN Quiescent Current in Shutdown - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 3 (table p3-t1, page 3)
| Parameter | Conditions | Min | Typ | Max | Unit |
|---|---|---|---|---|---|
| VIN Quiescent Current in Shutdown | VEN = 0.1V |  | 1 | 2 | μA |
```

**[2] ltc3310s-3310s-1-3310-3310-1.pdf - page 3 - table** (score 0.6667)

```
[ELECTRICAL CHARACTERISTICS]
VIN Quiescent Current - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 3 (table p3-t1, page 3)
| Parameter | Conditions | Min | Typ | Max | Unit |
|---|---|---|---|---|---|
| VIN Quiescent Current | (Note 4) |  | 1.3 | 2.0 | mA |
```

**[3] ltc3310s-3310s-1-3310-3310-1.pdf - page 4 - table** (score 0.2976)

```
[ELECTRICAL CHARACTERISTICS]
Top Switch Current Limit - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 4 (table p4-t1, page 4)
| Parameter | Conditions | Min | Typ | Max | Unit |
|---|---|---|---|---|---|
| Top Switch Current Limit | VOUT /VIN ≤ 0.2, Current Out of SW \| over full operating temperature range | 14.5 | 16 | 18.5 | A |
```

**[4] ltc3310s-3310s-1-3310-3310-1.pdf - page 19 - text** (score 0.2576)

```
[APPLICATIONS INFORMATION > Enable Threshold Programming]
The LTC3310S has a precision threshold enable pin to enable or disable switching. When forced low, the LTC3310S enters a low current shutdown mode. The rising threshold of the EN comparator is 400mV, with 60mV of hysteresis. Connect the EN pin to V if the shut- IN down feature is not used. Adding a resistor divider from V to EN programs the LTC3310S to regulate the output IN only when V is above a desired voltage (see the Block IN Diagram). Typically, this threshold, V , is used in sit- IN(EN) uations where the input supply is current limited or has a relatively high source resistance. A switching regulator draws constant power from the source, so source current increases as source voltage drops. This looks like a negative resistance load to the source and can cause the source to current limit or latch low under low source voltage conditions. The V threshold prevents the regulator from IN(EN) operating at source voltages where problems may occur.
```

**[5] ltc3310s-3310s-1-3310-3310-1.pdf - page 12 - text** (score 0.2500)

```
[OPERATION > Voltage Regulation]
The internal error amplifier senses the difference between this feedback voltage and a 0.5V AGND referenced voltage.

This scheme overcomes any ground offsets between local ground and remote output ground, resulting in a more accurate output voltage.

The LTC3310S allows for remote output ground deviations as much as ±100mV with respect to local ground.

If the EN pin is low, the LTC3310S is shut down and in a low quiescent current state.

When the EN pin is above its threshold, the switching regulator will be enabled.
```

### Compare dense vs sparse vs hybrid

In [19]:
def compare(question, top_k=TOP_K):
    cols = {}
    for mode in ("dense", "sparse", "hybrid"):
        cols[mode] = [f"p{p.payload['page']} {p.payload['type']}: {oneline(p.payload['text'], 70)}"
                      for p in search(question, mode=mode, top_k=top_k)]
        cols[mode] += [""] * (top_k - len(cols[mode]))
    df = pd.DataFrame(cols, index=range(1, top_k + 1))
    df.index.name = "rank"
    return df


compare(QUESTION)

,dense,sparse,hybrid
rank,,,
1,p3 table: [ELECTRICAL CHARACTERISTICS] VIN Quiescent Current in Shutdown - LTC33,p3 table: [ELECTRICAL CHARACTERISTICS] VIN Quiescent Current in Shutdown - LTC33,p3 table: [ELECTRICAL CHARACTERISTICS] VIN Quiescent Current in Shutdown - LTC33
2,p3 table: [ELECTRICAL CHARACTERISTICS] VIN Quiescent Current - LTC3310S/LTC3310S,p3 table: [ELECTRICAL CHARACTERISTICS] VIN Quiescent Current - LTC3310S/LTC3310S,p3 table: [ELECTRICAL CHARACTERISTICS] VIN Quiescent Current - LTC3310S/LTC3310S
3,p4 table: [ELECTRICAL CHARACTERISTICS] Top Switch Current Limit - LTC3310S/LTC33,p12 text: [OPERATION > Voltage Regulation] The internal error amplifier senses t,p4 table: [ELECTRICAL CHARACTERISTICS] Top Switch Current Limit - LTC3310S/LTC33
4,p4 table: [ELECTRICAL CHARACTERISTICS] Bottom Switch Current Limit (IVALLEYMAX ),p13 text: [OPERATION > Soft-Start/Tracking/Temperature Monitor] The soft-start t,p19 text: [APPLICATIONS INFORMATION > Enable Threshold Programming] The LTC3310S
5,p4 table: [ELECTRICAL CHARACTERISTICS] PGOOD Rising Threshold - LTC3310S/LTC3310,p19 text: [APPLICATIONS INFORMATION > Enable Threshold Programming] The LTC3310S,p4 table: [ELECTRICAL CHARACTERISTICS] Bottom Switch Current Limit (IVALLEYMAX )


### Try several questions

In [20]:
SAMPLE_QUESTIONS = [
    "What is the input voltage range?",
    "What is the switching frequency?",
    "Maximum output current of the regulator",
    "How do I configure the PGOOD pin?",
    "Absolute maximum junction temperature",
    "What package options are available?",
]

for q in SAMPLE_QUESTIONS:
    top = search(q, mode="hybrid", top_k=3)
    print(f"\nQ: {q}")
    for i, p in enumerate(top, 1):
        print(f"  {i}. p{p.payload['page']:<3} {p.payload['type']:<7} {oneline(p.payload['text'], 110)}")


Q: What is the input voltage range?
  1. p14  text    [APPLICATIONS INFORMATION > Operating Frequency Selection and Trade-Offs] Selection of the operating frequency
  2. p3   table   [ELECTRICAL CHARACTERISTICS] Regulated Output Voltage - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 
  3. p13  text    [OPERATION > Dropout Operation] As the input supply voltage approaches the output voltage, the duty cycle incr

Q: What is the switching frequency?
  1. p14  text    [APPLICATIONS INFORMATION > Operating Frequency Selection and Trade-Offs] Selection of the operating frequency
  2. p4   table   [ELECTRICAL CHARACTERISTICS] Switching Frequency - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 4 (ta
  3. p14  text    [APPLICATIONS INFORMATION > Setting the Switching Frequency] The LTC3310S uses a constant frequency PWM archit

Q: Maximum output current of the regulator
  1. p15  text    [APPLICATIONS INFORMATION > Inductor Selection and Maximum Output Current] where I is the 

## 6. Build a cited context for an LLM (optional answer generation)

`build_context` numbers the retrieved chunks so an answer can cite `[1]`, `[2]` ... with file and page.
The generation cell runs only if `OPENAI_API_KEY` is set; otherwise it prints the prompt.

In [21]:
def build_context(points):
    blocks = []
    for i, p in enumerate(points, 1):
        pl = p.payload
        blocks.append(f"[{i}] {pl['file_name']} | page {pl['page']} | {pl['section_path']}\n{pl['text']}")
    return "\n\n".join(blocks)


SYSTEM_PROMPT = (
    "You answer questions about semiconductor datasheets using ONLY the numbered sources provided. "
    "Quote numeric values exactly with their units and test conditions (min/typ/max). "
    "Cite sources like [1]. If the sources do not contain the answer, say so - do not guess."
)

context = build_context(hits)
user_prompt = f"Sources:\n{context}\n\nQuestion: {QUESTION}"
print(user_prompt[:3000])

Sources:
[1] ltc3310s-3310s-1-3310-3310-1.pdf | page 3 | ELECTRICAL CHARACTERISTICS
[ELECTRICAL CHARACTERISTICS]
VIN Quiescent Current in Shutdown - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 3 (table p3-t1, page 3)
| Parameter | Conditions | Min | Typ | Max | Unit |
|---|---|---|---|---|---|
| VIN Quiescent Current in Shutdown | VEN = 0.1V |  | 1 | 2 | μA |

[2] ltc3310s-3310s-1-3310-3310-1.pdf | page 3 | ELECTRICAL CHARACTERISTICS
[ELECTRICAL CHARACTERISTICS]
VIN Quiescent Current - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 3 (table p3-t1, page 3)
| Parameter | Conditions | Min | Typ | Max | Unit |
|---|---|---|---|---|---|
| VIN Quiescent Current | (Note 4) |  | 1.3 | 2.0 | mA |

[3] ltc3310s-3310s-1-3310-3310-1.pdf | page 4 | ELECTRICAL CHARACTERISTICS
[ELECTRICAL CHARACTERISTICS]
Top Switch Current Limit - LTC3310S/LTC3310S-1/LTC3310/LTC3310-1 - Table on page 4 (table p4-t1, page 4)
| Parameter | Conditions | Min | Typ | Max | Unit |
|---|---|---|---|---|---

In [22]:
OPENAI_MODEL = "gpt-5-mini"   # reasoning model: no temperature setting; the budget covers hidden reasoning + answer

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI

    llm = OpenAI()
    resp = llm.chat.completions.create(
        model=OPENAI_MODEL, reasoning_effort="medium", max_completion_tokens=4000,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}],
    )
    display(Markdown(resp.choices[0].message.content))
else:
    print("OPENAI_API_KEY not set - skipping generation. The retrieved context above is ready to send to any LLM.")

VIN Quiescent Current in Shutdown — Conditions: VEN = 0.1V — Typ: "1 μA", Max: "2 μA" (no Min listed) [1].  

(EN forced low puts the LTC3310S into the low‑current shutdown state [4][5].)

In [23]:
# One-call helper for interactive use
def ask(question, mode="hybrid", top_k=TOP_K, **filters):
    pts = search(question, mode=mode, top_k=top_k, **filters)
    show(pts, full_text=False)
    return pts

# Example: ask("Quiescent current of LTC3310S-1", chunk_type="table")

In [24]:
# Release the store (needed in "local" mode before re-running notebook 1)
client.close()